In [1]:
import os
import io
import time
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from googleapiclient.http import MediaIoBaseDownload
import pickle
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

def download_file(service, file_obj, parent_path, max_retries=3):
    import time
    import socket
    import traceback
    from googleapiclient.http import MediaIoBaseDownload

    file_id = file_obj['id']
    file_name = file_obj['name']
    file_path = os.path.join(parent_path, file_name)

    if os.path.exists(file_path):
        print(f"✅ Skipped (already exists): {file_name}")
        return file_name, 0, os.path.getsize(file_path)

    attempt = 0
    while attempt < max_retries:
        try:
            request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
            with io.FileIO(file_path, 'wb') as fh:
                downloader = MediaIoBaseDownload(fh, request)
                done = False
                start = time.time()

                while not done:
                    try:
                        status, done = downloader.next_chunk()
                    except socket.timeout:
                        raise TimeoutError("Chunk read timed out")

                duration = time.time() - start
                size = os.path.getsize(file_path)
                return file_name, duration, size

        except (socket.timeout, TimeoutError):
            attempt += 1
            print(f"⏱️ Timeout on {file_name}, retrying ({attempt}/{max_retries})...")
            time.sleep(2)
        except Exception as e:
            print(f"❌ Error on {file_name}: {e}")
            traceback.print_exc()
            break

        # Cleanup if retrying
        if os.path.exists(file_path):
            try:
                os.remove(file_path)
            except Exception:
                pass

    print(f"❌ Failed after {max_retries} attempts: {file_name}")
    return file_name, 0, 0


In [3]:
def list_folder_contents(service, folder_id, parent_path='PhdFiles', drive_id=None, file_counter=None):
    import sys
    from googleapiclient.http import MediaIoBaseDownload

    os.makedirs(parent_path, exist_ok=True)

    if file_counter is None:
        file_counter = {'total': 0, 'current': 0}

    query = f"'{folder_id}' in parents and trashed = false"
    list_kwargs = {
        'q': query,
        'fields': "files(id, name, mimeType)",
        'supportsAllDrives': True,
        'includeItemsFromAllDrives': True
    }

    if drive_id:
        list_kwargs['corpora'] = 'drive'
        list_kwargs['driveId'] = drive_id
    else:
        list_kwargs['corpora'] = 'user'

    # Only calculate total file count once
    if file_counter['total'] == 0:
        def count_files(folder_id):
            count = 0
            sub_kwargs = list_kwargs.copy()
            sub_kwargs['q'] = f"'{folder_id}' in parents and trashed = false"
            sub_results = service.files().list(**sub_kwargs).execute()
            for item in sub_results.get('files', []):
                if item['mimeType'] == 'application/vnd.google-apps.folder':
                    count += count_files(item['id'])
                else:
                    count += 1
            return count

        file_counter['total'] = count_files(folder_id)
        print(f"\n🔢 Total files to download: {file_counter['total']}\n")

    results = service.files().list(**list_kwargs).execute()
    items = results.get('files', [])

    if not items:
        print("No files found in this folder.")
    else:
        for item in items:
            name = item['name']
            file_id = item['id']
            mime_type = item['mimeType']

            if mime_type == 'application/vnd.google-apps.folder':
                new_path = os.path.join(parent_path, name)
                list_folder_contents(service, file_id, new_path, drive_id, file_counter)
            else:
                file_counter['current'] += 1
                print(f"\n📥 [{file_counter['current']} / {file_counter['total']}] Downloading: {name}")
                file_path = os.path.join(parent_path, name)

                request = service.files().get_media(fileId=file_id, supportsAllDrives=True)
                with io.FileIO(file_path, 'wb') as fh:
                    downloader = MediaIoBaseDownload(fh, request)
                    done = False
                    while not done:
                        status, done = downloader.next_chunk()
                        progress = int(status.progress() * 100)
                        sys.stdout.write(f"\r    {progress}% complete")
                        sys.stdout.flush()
                print()  # Newline after progress

In [5]:
def collect_files(service, folder_id, parent_path='downloads', drive_id=None):
    os.makedirs(parent_path, exist_ok=True)
    files_to_download = []

    query = f"'{folder_id}' in parents and trashed = false"
    list_kwargs = {
        'q': query,
        'fields': "files(id, name, mimeType)",
        'supportsAllDrives': True,
        'includeItemsFromAllDrives': True,
        'corpora': 'drive' if drive_id else 'user'
    }
    if drive_id:
        list_kwargs['driveId'] = drive_id

    response = service.files().list(**list_kwargs).execute()
    items = response.get('files', [])

    for item in items:
        name = item['name']
        mime_type = item['mimeType']
        file_id = item['id']

        if mime_type == 'application/vnd.google-apps.folder':
            new_path = os.path.join(parent_path, name)
            sub_files = collect_files(service, file_id, new_path, drive_id)
            files_to_download.extend(sub_files)
        else:
            files_to_download.append({'id': file_id, 'name': name, 'path': parent_path})

    return files_to_download

In [7]:
def download_all_files(service, files_to_download, max_workers=4):
    print(f"\n🔄 Starting downloads ({len(files_to_download)} files)...\n")
    total_size = 0
    total_time = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(download_file, service, f, f['path'])
            for f in files_to_download
        ]

        for future in as_completed(futures):
            try:
                name, duration, size = future.result()
                if duration > 0:
                    speed_mbps = (size * 8) / (duration * 1_000_000)
                    print(f"📥 {name} - {size/1_000:.1f} KB in {duration:.2f}s ({speed_mbps:.2f} Mbps)")
                total_size += size
                total_time += duration
            except Exception as e:
                print(f"❌ Error: {e}")

    if total_time > 0:
        avg_speed = (total_size * 8) / (total_time * 1_000_000)
        print(f"\n✅ All done. {len(files_to_download)} files, {total_size/1_000_000:.2f} MB total, avg {avg_speed:.2f} Mbps")

In [9]:
def authenticate():
    import pickle
    from google.auth.transport.requests import Request
    from google_auth_oauthlib.flow import InstalledAppFlow

    SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
    creds = None

    if os.path.exists('token.pickle'):
        with open('token.pickle', 'rb') as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_console()  # ✅ Jupyter-friendly auth
        with open('token.pickle', 'wb') as token:
            pickle.dump(creds, token)

    return creds

In [11]:
import ssl
import urllib3

# Override SSL context for httplib2 (used under the hood by Google API)
from googleapiclient.http import set_user_agent

try:
    ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    ssl_context.check_hostname = True
    ssl_context.verify_mode = ssl.CERT_REQUIRED
except Exception as e:
    print("⚠️ Could not set up custom SSL context:", e)

In [15]:
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
FOLDER_ID = '18BMH-PhTZNiN8rbXyqZvz5J9qRGrRqbA'  # just the ID
DRIVE_ID = '0AHMeY0Fo1d79Uk9PVA'
CUSTOM_DOWNLOAD_PATH = r'C:\Users\Ry\Documents\PhD Thesis Writing\Data'

creds = authenticate()
service = build('drive', 'v3', credentials=creds)

files = collect_files(service, FOLDER_ID, CUSTOM_DOWNLOAD_PATH, DRIVE_ID)
download_all_files(service, files, max_workers=4)


🔄 Starting downloads (496 files)...

✅ Skipped (already exists): fiber_fiber_interferometer_measurement_weekend.csv
✅ Skipped (already exists): Powermeterlog2025-05-30_16-44-45-PM.txt
✅ Skipped (already exists): ps3.npy
✅ Skipped (already exists): ps2.npy
✅ Skipped (already exists): ps.npy
✅ Skipped (already exists): vs.npy
✅ Skipped (already exists): fiber_fiber_interferometer_measurement_long.csv
✅ Skipped (already exists): fiber_fiber_interferometer_measurement5.csv
✅ Skipped (already exists): fiber_fiber_interferometer_measurement4.csv
✅ Skipped (already exists): fiber_fiber_interferometer_measurement3.csv
✅ Skipped (already exists): fiber_fiber_interferometer_measurement2.csv
✅ Skipped (already exists): fiber_fiber_interferometer_measurement1.csv
✅ Skipped (already exists): 1km_1km_interferometer_long.csv
✅ Skipped (already exists): measurement_24_07_25_1km_ramp1_lvns.npy
✅ Skipped (already exists): measurement_24_07_25_1km_ramp1_lvts.npy
✅ Skipped (already exists): measurement_2

# 🔧 Proxy-safe Google Drive downloader (patched 2025-08-20 20:17 UTC)

This section adds a **proxy-resistant** download flow that fixes SSL errors like:
`[SSL: WRONG_VERSION_NUMBER] wrong version number (_ssl.c:2653)`

**How to use:**
1. Run the **Install/Update deps** cell (optional if you already have these packages).
2. Run the **Disable proxies (optional but recommended)** cell.
3. Fill in your `FOLDER_ID` (or a `FILE_ID`) in the **Configure & run** cell and execute it.
4. Downloads will be saved to your chosen folder. Native Google Sheets will be **exported to CSV** automatically.


In [13]:
# 1) Install / Update dependencies (optional)
import sys, subprocess
def _pip_install(pkgs):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])
    except Exception as e:
        print("Pip install failed (you can ignore if already installed):", e)

_pip_install([
    "google-api-python-client",
    "google-auth",
    "google-auth-oauthlib",
    "requests",
    "urllib3",
    "certifi",
])

KeyboardInterrupt: 

In [13]:
# 2) Disable proxies (optional but recommended to avoid SSL issues)
import os
print("Before:")
print(" HTTP_PROXY =", os.environ.get("HTTP_PROXY") or os.environ.get("http_proxy"))
print(" HTTPS_PROXY =", os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy"))
print(" NO_PROXY   =", os.environ.get("NO_PROXY") or os.environ.get("no_proxy"))

for k in ["HTTP_PROXY","http_proxy","HTTPS_PROXY","https_proxy"]:
    os.environ.pop(k, None)
os.environ["NO_PROXY"] = "*"
os.environ["no_proxy"] = "*"

print("\nAfter:")
print(" HTTP_PROXY =", os.environ.get("HTTP_PROXY") or os.environ.get("http_proxy"))
print(" HTTPS_PROXY =", os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy"))
print(" NO_PROXY   =", os.environ.get("NO_PROXY") or os.environ.get("no_proxy"))

Before:
 HTTP_PROXY = None
 HTTPS_PROXY = None
 NO_PROXY   = None

After:
 HTTP_PROXY = None
 HTTPS_PROXY = None
 NO_PROXY   = *


In [15]:
# 3) Auth and proxy-safe download utilities
from __future__ import annotations
import io, os, time, pathlib, requests, typing as _t
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import AuthorizedSession, Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

def get_creds(creds_path: str = "credentials.json", token_path: str = "token.json") -> Credentials:
    creds = None
    if pathlib.Path(token_path).exists():
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(creds_path, SCOPES)
            creds = flow.run_local_server(port=0)
        pathlib.Path(token_path).write_text(creds.to_json(), encoding="utf-8")
    return creds

def get_authed_session(creds: Credentials) -> AuthorizedSession:
    # Ignore system proxies to avoid SSL handshake issues
    sess = requests.Session()
    sess.trust_env = False
    # Add a default timeout for all requests
    original_request = sess.request
    def _with_timeout(method, url, **kwargs):
        kwargs.setdefault("timeout", 60)
        return original_request(method, url, **kwargs)
    sess.request = _with_timeout
    return AuthorizedSession(creds, session=sess)

def _stream_download(authed: AuthorizedSession, url: str, dest_path: str, max_retries: int = 3):
    p = pathlib.Path(dest_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    backoff = 1.5
    for attempt in range(1, max_retries + 1):
        try:
            with authed.get(url, stream=True) as r:
                r.raise_for_status()
                with p.open("wb") as fh:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        if chunk:
                            fh.write(chunk)
            print(f"✅ Saved to {p}")
            return
        except requests.exceptions.SSLError as e:
            print(f"❌ SSL error (attempt {attempt}/{max_retries}): {e}")
        except requests.RequestException as e:
            print(f"❌ HTTP error (attempt {attempt}/{max_retries}): {e}")
        if attempt < max_retries:
            time.sleep(backoff)
            backoff *= 2
    raise RuntimeError(f"Failed after {max_retries} attempts: {dest_path}")

def drive_service(creds: Credentials):
    return build("drive", "v3", credentials=creds)

def list_files_in_folder(folder_id: str):
    svc = drive_service(get_creds())
    items, token = [], None
    while True:
        resp = svc.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id,name,mimeType)",
            pageToken=token
        ).execute()
        items.extend(resp.get("files", []))
        token = resp.get("nextPageToken")
        if not token:
            break
    return items

GOOGLE_MIME_FOLDER = "application/vnd.google-apps.folder"
GOOGLE_MIME_SHEET  = "application/vnd.google-apps.spreadsheet"

def download_file_by_id(file_id: str, dest_path: str, mime_type_if_export: str | None = None):
    creds = get_creds()
    authed = get_authed_session(creds)
    if mime_type_if_export:
        url = f"https://www.googleapis.com/drive/v3/files/{file_id}/export?mimeType={mime_type_if_export}"
    else:
        url = f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media"
    _stream_download(authed, url, dest_path)

def smart_download_item(item: dict, out_dir: str):
    name = item["name"]
    fid  = item["id"]
    mt   = item["mimeType"]
    dest = str(pathlib.Path(out_dir, name))
    if mt == GOOGLE_MIME_FOLDER:
        print(f"📁 Skipping folder (use recursion if needed): {name}")
        return
    try:
        # Export Google Sheets -> CSV if filename ends with .csv, else to .xlsx
        if mt == GOOGLE_MIME_SHEET:
            if name.lower().endswith(".csv"):
                download_file_by_id(fid, dest, mime_type_if_export="text/csv")
            else:
                if not name.lower().endswith(".xlsx"):
                    dest += ".xlsx"
                download_file_by_id(fid, dest, mime_type_if_export="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")
        elif mt.startswith("application/vnd.google-apps."):
            print(f"↪ Skipping native Google file type ({mt}) for {name}. Add an export if needed.")
        else:
            download_file_by_id(fid, dest)
    except Exception as e:
        print(f"❌ Error on {name}: {e}")

def smart_download_folder(folder_id: str, out_dir: str):
    items = list_files_in_folder(folder_id)
    print(f"Found {len(items)} item(s) in folder.")
    for it in items:
        smart_download_item(it, out_dir)

In [17]:
# Install & import tqdm safely (get the callable as _tqdm)
try:
    from tqdm.auto import tqdm as _tqdm
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "tqdm"])
    from tqdm.auto import tqdm as _tqdm

# If you previously bound a variable named `tqdm`, unshadow it:
try:
    del tqdm  # noqa: F821
except NameError:
    pass

In [19]:
# Progress-enabled download + recursive sync (uses _tqdm for bars, print for messages)
from pathlib import Path

# -- Override the low-level streaming download to show a bytes progress bar --
def _stream_download(authed, url: str, dest_path: str, max_retries: int = 3):
    p = Path(dest_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    backoff = 1.5
    for attempt in range(1, max_retries + 1):
        try:
            with authed.get(url, stream=True) as r:
                r.raise_for_status()
                total = r.headers.get("Content-Length")
                total = int(total) if total and total.isdigit() else None
                with p.open("wb") as fh, _tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=p.name,
                    leave=True,
                ) as bar:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            fh.write(chunk)
                            bar.update(len(chunk))
            print(f"✔ Saved: {p}")
            return
        except Exception as e:
            print(f"Retry {attempt}: {e}")
        if attempt < max_retries:
            import time
            time.sleep(backoff)
            backoff *= 2
    raise RuntimeError(f"Failed after {max_retries} attempts: {dest_path}")

# -- Helpers for listing + deciding export (requires GOOGLE_MIME_FOLDER/GOOGLE_MIME_SHEET & drive helpers already defined) --
def list_children_all_drives(svc, folder_id: str):
    items, token = [], None
    while True:
        resp = svc.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id,name,mimeType)",
            pageToken=token,
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
            corpora="allDrives",
        ).execute()
        items.extend(resp.get("files", []))
        token = resp.get("nextPageToken")
        if not token:
            break
    return items

def _decide_export_and_dest(name: str, mime_type: str, local_dir: str):
    p = Path(local_dir, name)
    if mime_type == GOOGLE_MIME_SHEET:
        if name.lower().endswith(".csv"):
            return "text/csv", p
        if not name.lower().endswith(".xlsx"):
            p = p.with_suffix(p.suffix + ".xlsx" if p.suffix else ".xlsx")
        return "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", p
    return None, p

# -- Progress-enabled recursive sync (preserves folder structure & skips existing files) --
def sync_drive_folder(folder_id: str, local_dir: str):
    svc = drive_service(get_creds())
    Path(local_dir).mkdir(parents=True, exist_ok=True)

    children = list_children_all_drives(svc, folder_id)

    file_items = [it for it in children if it["mimeType"] != GOOGLE_MIME_FOLDER]
    folder_name = Path(local_dir).name or str(local_dir)
    print(f"📂 {folder_name}: {len(children)} item(s) ({len(file_items)} files)")
    with _tqdm(total=len(file_items), desc=f"{folder_name}", unit="file", leave=True) as pbar:
        for it in children:
            name, fid, mt = it["name"], it["id"], it["mimeType"]
            if mt == GOOGLE_MIME_FOLDER:
                # Recurse into subfolder (its own bar will show when that call runs)
                sync_drive_folder(fid, str(Path(local_dir, name)))
                continue

            export_mime, dest = _decide_export_and_dest(name, mt, local_dir)

            # Skip if already downloaded
            if dest.exists():
                pbar.update(1)
                print(f"⏭️  Exists, skipping: {dest}")
                continue

            try:
                if export_mime:
                    download_file_by_id(fid, str(dest), mime_type_if_export=export_mime)
                elif mt.startswith("application/vnd.google-apps."):
                    print(f"↪ Skipping unsupported native type ({mt}) for {name}")
                else:
                    download_file_by_id(fid, str(dest))
                pbar.update(1)
            except Exception as e:
                pbar.update(1)
                print(f"❌ Error on {name}: {e}")

In [21]:
# Add counters + live updates in the progress bar + final summary
from dataclasses import dataclass
from pathlib import Path

try:
    _tqdm  # from tqdm import tqdm as _tqdm  (should already be defined from earlier step)
except NameError:
    from tqdm import tqdm as _tqdm  # fallback to text-mode tqdm if not imported yet

@dataclass
class SyncStats:
    files_checked: int = 0
    downloaded: int = 0
    skipped_existing: int = 0
    skipped_native: int = 0
    errors: int = 0

def sync_drive_folder(folder_id: str, local_dir: str, *, stats: SyncStats | None = None, svc=None, _root: bool = True):
    """
    Recursively sync a Drive folder to `local_dir`, preserving structure.
    Shows a per-folder progress bar and maintains global counters across recursion.
    """
    if stats is None:
        stats = SyncStats()
    if svc is None:
        svc = drive_service(get_creds())  # reuse the same service for all recursion

    Path(local_dir).mkdir(parents=True, exist_ok=True)

    children = list_children_all_drives(svc, folder_id)
    file_items = [it for it in children if it["mimeType"] != GOOGLE_MIME_FOLDER]
    folder_name = Path(local_dir).name or str(local_dir)
    print(f"📂 {folder_name}: {len(children)} item(s) ({len(file_items)} files)")

    with _tqdm(total=len(file_items), desc=f"{folder_name}", unit="file", leave=True) as pbar:
        for it in children:
            name, fid, mt = it["name"], it["id"], it["mimeType"]

            if mt == GOOGLE_MIME_FOLDER:
                # Recurse into the subfolder
                sync_drive_folder(fid, str(Path(local_dir, name)), stats=stats, svc=svc, _root=False)
                continue

            # We're checking a file (non-folder)
            stats.files_checked += 1

            export_mime, dest = _decide_export_and_dest(name, mt, local_dir)

            # Skip if already downloaded
            if dest.exists():
                stats.skipped_existing += 1
                pbar.update(1)
                pbar.set_postfix(checked=stats.files_checked, dl=stats.downloaded, skip=stats.skipped_existing, err=stats.errors)
                print(f"⏭️  Exists, skipping: {dest}")
                continue

            try:
                if export_mime:
                    download_file_by_id(fid, str(dest), mime_type_if_export=export_mime)
                    stats.downloaded += 1
                elif mt.startswith("application/vnd.google-apps."):
                    # Unsupported native Google type (Docs/Slides/Drawings) – count as skipped
                    stats.skipped_native += 1
                    print(f"↪ Skipping unsupported native type ({mt}) for {name}")
                else:
                    download_file_by_id(fid, str(dest))
                    stats.downloaded += 1
            except Exception as e:
                stats.errors += 1
                print(f"❌ Error on {name}: {e}")

            pbar.update(1)
            pbar.set_postfix(checked=stats.files_checked, dl=stats.downloaded, skip=stats.skipped_existing, err=stats.errors)

    if _root:
        print(
            "\nSummary:"
            f"\n  • Files checked:      {stats.files_checked}"
            f"\n  • Downloaded:         {stats.downloaded}"
            f"\n  • Skipped (existing): {stats.skipped_existing}"
            f"\n  • Skipped (native):   {stats.skipped_native}"
            f"\n  • Errors:             {stats.errors}"
        )
    return stats


In [23]:
FOLDER_ID = '1bZYi9zgWendn5ZfqFCbigJ-fXuh7lb5_'
FILE_ID   = None
OUT_DIR   = r'D:\PhD Thesis Writing\Data\2025-11-26'
sync_drive_folder(FOLDER_ID, OUT_DIR)

📂 2025-11-26: 6 item(s) (0 files)


2025-11-26: 0file [00:00, ?file/s]

📂 Day2LongNoFibre: 36 item(s) (36 files)


Day2LongNoFibre:   0%|          | 0/36 [00:00<?, ?file/s]

❌ Error on Day_2_1MhzGate_107uWperArm_Rmp1000Hz_voaramp_20251126T184351+0000.npz: __init__() got an unexpected keyword argument 'session'
❌ Error on Day_2_1MhzGate_107uWperArm_Rmp1000Hz_voaramp_20251126T184351+0000.csv: __init__() got an unexpected keyword argument 'session'
❌ Error on Day_2_1MhzGate_107uWperArm_Rmp1000Hz_voaramp_20251126T184351+0000_meta.txt: __init__() got an unexpected keyword argument 'session'
❌ Error on Day_2_1MhzGate_107uWperArm_Rmp100Hz_voaramp_20251126T184025+0000_meta.txt: __init__() got an unexpected keyword argument 'session'
❌ Error on Day_2_1MhzGate_107uWperArm_Rmp100Hz_voaramp_20251126T184025+0000.csv: __init__() got an unexpected keyword argument 'session'
❌ Error on Day_2_1MhzGate_107uWperArm_Rmp100Hz_voaramp_20251126T184025+0000.npz: __init__() got an unexpected keyword argument 'session'
❌ Error on Day_2_1MhzGate_107uWperArm_Rmp10Hz_voaramp_20251126T183658+0000.npz: __init__() got an unexpected keyword argument 'session'
❌ Error on Day_2_1MhzGate_107

Medium2LongNoFibre:   0%|          | 0/36 [00:00<?, ?file/s]

... (6 outputs left out)

❌ Error on Medium_2_1MhzGate_130uWperArm_Rmp1000Hz_voaramp_20251126T215824+0000.npz: __init__() got an unexpected keyword argument 'session'
❌ Error on Medium_2_1MhzGate_130uWperArm_Rmp1000Hz_voaramp_20251126T215824+0000_meta.txt: __init__() got an unexpected keyword argument 'session'
❌ Error on Medium_2_1MhzGate_130uWperArm_Rmp1000Hz_voaramp_20251126T215824+0000.csv: __init__() got an unexpected keyword argument 'session'
❌ Error on Medium_2_1MhzGate_130uWperArm_Rmp100Hz_voaramp_20251126T215457+0000_meta.txt: __init__() got an unexpected keyword argument 'session'
❌ Error on Medium_2_1MhzGate_130uWperArm_Rmp100Hz_voaramp_20251126T215457+0000.csv: __init__() got an unexpected keyword argument 'session'
❌ Error on Medium_2_1MhzGate_130uWperArm_Rmp100Hz_voaramp_20251126T215457+0000.npz: __init__() got an unexpected keyword argument 'session'
❌ Error on Medium_2_1MhzGate_130uWperArm_Rmp10Hz_voaramp_20251126T215130+0000.npz: __init__() got an unexpected keyword argument 'session'
❌ Error 

Night2LongNoFibre:   0%|          | 0/42 [00:00<?, ?file/s]

❌ Error on Night_2_1MhzGate_195uWperArm_Rmp1000Hz_voaramp_20251126T161134+0000.npz: __init__() got an unexpected keyword argument 'session'
❌ Error on Night_2_1MhzGate_195uWperArm_Rmp1000Hz_voaramp_20251126T161134+0000_meta.txt: __init__() got an unexpected keyword argument 'session'
❌ Error on Night_2_1MhzGate_195uWperArm_Rmp1000Hz_voaramp_20251126T161134+0000.csv: __init__() got an unexpected keyword argument 'session'
❌ Error on Night_2_1MhzGate_195uWperArm_Rmp100Hz_voaramp_20251126T160800+0000.csv: __init__() got an unexpected keyword argument 'session'
❌ Error on Night_2_1MhzGate_195uWperArm_Rmp100Hz_voaramp_20251126T160800+0000_meta.txt: __init__() got an unexpected keyword argument 'session'
❌ Error on Night_2_1MhzGate_195uWperArm_Rmp100Hz_voaramp_20251126T160800+0000.npz: __init__() got an unexpected keyword argument 'session'
❌ Error on Night_2_1MhzGate_195uWperArm_Rmp10Hz_voaramp_20251126T155932+0000.npz: __init__() got an unexpected keyword argument 'session'
❌ Error on Nigh

SyncStats(files_checked=228, downloaded=0, skipped_existing=0, skipped_native=0, errors=228)

In [27]:
# Text-mode progress bars (no ipywidgets required)
# Paste this cell, run it, then run your Configure & run cell.
try:
    from tqdm import tqdm as _tqdm   # <- force console/text progress bar
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "tqdm"])
    from tqdm import tqdm as _tqdm

from pathlib import Path

# --- bytes progress for each file ---
def _stream_download(authed, url: str, dest_path: str, max_retries: int = 3):
    p = Path(dest_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    backoff = 1.5
    for attempt in range(1, max_retries + 1):
        try:
            with authed.get(url, stream=True) as r:
                r.raise_for_status()
                total = r.headers.get("Content-Length")
                total = int(total) if (total and total.isdigit()) else None
                with p.open("wb") as fh, _tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=p.name,
                    leave=True,
                ) as bar:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            fh.write(chunk)
                            bar.update(len(chunk))
            print(f"✔ Saved: {p}")
            return
        except Exception as e:
            print(f"Retry {attempt}: {e}")
        if attempt < max_retries:
            import time
            time.sleep(backoff)
            backoff *= 2
    raise RuntimeError(f"Failed after {max_retries} attempts: {dest_path}")

# --- helpers (expect GOOGLE_MIME_FOLDER/GOOGLE_MIME_SHEET + drive helpers already defined) ---
def list_children_all_drives(svc, folder_id: str):
    items, token = [], None
    while True:
        resp = svc.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id,name,mimeType)",
            pageToken=token,
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
            corpora="allDrives",
        ).execute()
        items.extend(resp.get("files", []))
        token = resp.get("nextPageToken")
        if not token:
            break
    return items

def _decide_export_and_dest(name: str, mime_type: str, local_dir: str):
    p = Path(local_dir, name)
    if mime_type == GOOGLE_MIME_SHEET:
        if name.lower().endswith(".csv"):
            return "text/csv", p
        if not name.lower().endswith(".xlsx"):
            p = p.with_suffix(p.suffix + ".xlsx" if p.suffix else ".xlsx")
        return "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", p
    return None, p

# --- recursive sync with per-folder file counter & skip-existing ---
def sync_drive_folder(folder_id: str, local_dir: str):
    svc = drive_service(get_creds())
    Path(local_dir).mkdir(parents=True, exist_ok=True)

    children = list_children_all_drives(svc, folder_id)
    file_items = [it for it in children if it["mimeType"] != GOOGLE_MIME_FOLDER]
    folder_name = Path(local_dir).name or str(local_dir)
    print(f"📂 {folder_name}: {len(children)} item(s) ({len(file_items)} files)")

    with _tqdm(total=len(file_items), desc=f"{folder_name}", unit="file", leave=True) as pbar:
        for it in children:
            name, fid, mt = it["name"], it["id"], it["mimeType"]

            if mt == GOOGLE_MIME_FOLDER:
                # recurse into subfolder
                sync_drive_folder(fid, str(Path(local_dir, name)))
                continue

            export_mime, dest = _decide_export_and_dest(name, mt, local_dir)

            # skip already-downloaded files
            if dest.exists():
                pbar.update(1)
                print(f"⏭️  Exists, skipping: {dest}")
                continue

            try:
                if export_mime:
                    download_file_by_id(fid, str(dest), mime_type_if_export=export_mime)
                elif mt.startswith("application/vnd.google-apps."):
                    print(f"↪ Skipping unsupported native type ({mt}) for {name}")
                else:
                    download_file_by_id(fid, str(dest))
                pbar.update(1)
            except Exception as e:
                pbar.update(1)
                print(f"❌ Error on {name}: {e}")

In [33]:
# Text-mode progress bars (no ipywidgets required)
# Paste this cell, run it, then run your Configure & run cell.
try:
    from tqdm import tqdm as _tqdm   # <- force console/text progress bar
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "tqdm"])
    from tqdm import tqdm as _tqdm

from pathlib import Path

# --- bytes progress for each file ---
def _stream_download(authed, url: str, dest_path: str, max_retries: int = 3):
    p = Path(dest_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    backoff = 1.5
    for attempt in range(1, max_retries + 1):
        try:
            with authed.get(url, stream=True) as r:
                r.raise_for_status()
                total = r.headers.get("Content-Length")
                total = int(total) if (total and total.isdigit()) else None
                with p.open("wb") as fh, _tqdm(
                    total=total,
                    unit="B",
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=p.name,
                    leave=True,
                ) as bar:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            fh.write(chunk)
                            bar.update(len(chunk))
            print(f"✔ Saved: {p}")
            return
        except Exception as e:
            print(f"Retry {attempt}: {e}")
        if attempt < max_retries:
            import time
            time.sleep(backoff)
            backoff *= 2
    raise RuntimeError(f"Failed after {max_retries} attempts: {dest_path}")

# --- helpers (expect GOOGLE_MIME_FOLDER/GOOGLE_MIME_SHEET + drive helpers already defined) ---
def list_children_all_drives(svc, folder_id: str):
    items, token = [], None
    while True:
        resp = svc.files().list(
            q=f"'{folder_id}' in parents and trashed=false",
            fields="nextPageToken, files(id,name,mimeType)",
            pageToken=token,
            includeItemsFromAllDrives=True,
            supportsAllDrives=True,
            corpora="allDrives",
        ).execute()
        items.extend(resp.get("files", []))
        token = resp.get("nextPageToken")
        if not token:
            break
    return items

def _decide_export_and_dest(name: str, mime_type: str, local_dir: str):
    p = Path(local_dir, name)
    if mime_type == GOOGLE_MIME_SHEET:
        if name.lower().endswith(".csv"):
            return "text/csv", p
        if not name.lower().endswith(".xlsx"):
            p = p.with_suffix(p.suffix + ".xlsx" if p.suffix else ".xlsx")
        return "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", p
    return None, p

# --- recursive sync with per-folder file counter & skip-existing ---
def sync_drive_folder(folder_id: str, local_dir: str):
    svc = drive_service(get_creds())
    Path(local_dir).mkdir(parents=True, exist_ok=True)

    children = list_children_all_drives(svc, folder_id)
    file_items = [it for it in children if it["mimeType"] != GOOGLE_MIME_FOLDER]
    folder_name = Path(local_dir).name or str(local_dir)
    print(f"📂 {folder_name}: {len(children)} item(s) ({len(file_items)} files)")

    with _tqdm(total=len(file_items), desc=f"{folder_name}", unit="file", leave=True) as pbar:
        for it in children:
            name, fid, mt = it["name"], it["id"], it["mimeType"]

            if mt == GOOGLE_MIME_FOLDER:
                # recurse into subfolder
                sync_drive_folder(fid, str(Path(local_dir, name)))
                continue

            export_mime, dest = _decide_export_and_dest(name, mt, local_dir)

            # skip already-downloaded files
            if dest.exists():
                pbar.update(1)
                print(f"⏭️  Exists, skipping: {dest}")
                continue

            try:
                if export_mime:
                    download_file_by_id(fid, str(dest), mime_type_if_export=export_mime)
                elif mt.startswith("application/vnd.google-apps."):
                    print(f"↪ Skipping unsupported native type ({mt}) for {name}")
                else:
                    download_file_by_id(fid, str(dest))
                pbar.update(1)
            except Exception as e:
                pbar.update(1)
                print(f"❌ Error on {name}: {e}")

In [35]:
# Sync with size-check & conditional redownload, with counters and text-mode progress bars.
from dataclasses import dataclass
from pathlib import Path
import os

try:
    _tqdm  # defined earlier as: from tqdm import tqdm as _tqdm
except NameError:
    from tqdm import tqdm as _tqdm  # fallback

@dataclass
class SyncStats:
    files_checked: int = 0
    downloaded: int = 0
    skipped_existing: int = 0
    skipped_native: int = 0
    redownloaded: int = 0
    errors: int = 0

def _get_remote_info(svc, file_id: str):
    """Fetch minimal metadata needed to compare sizes for non-native files."""
    return svc.files().get(
        fileId=file_id,
        fields="size, md5Checksum, mimeType, modifiedTime, name",
        supportsAllDrives=True,
    ).execute()

def _significant_size_diff(local_size: int, remote_size: int | None,
                           ratio_threshold: float, min_bytes: int) -> bool:
    if remote_size is None or remote_size <= 0:
        return False  # cannot compare
    diff = abs(local_size - remote_size)
    # Require both an absolute minimum and a proportional difference
    return diff >= min_bytes and (diff / max(remote_size, 1)) >= ratio_threshold

def sync_drive_folder(
    folder_id: str,
    local_dir: str,
    *,
    stats: SyncStats | None = None,
    svc=None,
    _root: bool = True,
    # Tuning knobs for "significant" size difference
    size_diff_ratio_threshold: float = 0.05,     # 5% or more difference
    size_diff_min_bytes: int = 1024,             # and at least 1 KiB difference
    refresh_native_if_zero: bool = True          # for exported Google Sheets, if local==0 bytes, redownload
):
    """
    Recursively sync a Drive folder to `local_dir`, preserving structure.
    If a local file exists and the remote size differs significantly (non-native files),
    delete the local and redownload.

    Parameters:
      size_diff_ratio_threshold: minimum fractional difference to consider "significant"
      size_diff_min_bytes: minimum absolute difference in bytes to consider "significant"
      refresh_native_if_zero: if True, re-export Google Sheets when local file is 0 bytes
    """
    if stats is None:
        stats = SyncStats()
    if svc is None:
        svc = drive_service(get_creds())  # reuse same service across recursion

    Path(local_dir).mkdir(parents=True, exist_ok=True)

    children = list_children_all_drives(svc, folder_id)
    file_items = [it for it in children if it["mimeType"] != GOOGLE_MIME_FOLDER]
    folder_name = Path(local_dir).name or str(local_dir)
    print(f"📂 {folder_name}: {len(children)} item(s) ({len(file_items)} files)")

    with _tqdm(total=len(file_items), desc=f"{folder_name}", unit="file", leave=True) as pbar:
        for it in children:
            name, fid, mt = it["name"], it["id"], it["mimeType"]

            if mt == GOOGLE_MIME_FOLDER:
                # Recurse into subfolder
                sync_drive_folder(
                    fid, str(Path(local_dir, name)),
                    stats=stats, svc=svc, _root=False,
                    size_diff_ratio_threshold=size_diff_ratio_threshold,
                    size_diff_min_bytes=size_diff_min_bytes,
                    refresh_native_if_zero=refresh_native_if_zero,
                )
                continue

            # Non-folder == file under consideration
            stats.files_checked += 1

            export_mime, dest = _decide_export_and_dest(name, mt, local_dir)

            # If a local file exists, decide whether to skip or redownload
            if dest.exists():
                # For Google-native exports, we don't have a reliable "remote size".
                # Optionally force redownload if the local file is clearly bad (0 bytes).
                if export_mime:
                    if refresh_native_if_zero and dest.stat().st_size == 0:
                        print(f"↻ Local is 0 bytes, redownloading (Google export): {dest.name}")
                        try:
                            dest.unlink()
                        except FileNotFoundError:
                            pass
                        # fall through to download below (no 'continue')
                    else:
                        stats.skipped_existing += 1
                        pbar.update(1)
                        pbar.set_postfix(checked=stats.files_checked, dl=stats.downloaded,
                                         re=stats.redownloaded, skip=stats.skipped_existing, err=stats.errors)
                        print(f"⏭️  Exists, skipping: {dest}")
                        continue
                else:
                    # Non-native file: compare remote size to local size
                    local_size = dest.stat().st_size
                    remote_size = None
                    try:
                        meta = _get_remote_info(svc, fid)
                        if "size" in meta and meta["size"] is not None:
                            remote_size = int(meta["size"])
                    except Exception as e:
                        # If we can't fetch size, safely skip size-based decision
                        print(f"ℹ️  Could not fetch remote size for {name}: {e}")

                    if _significant_size_diff(local_size, remote_size,
                                              size_diff_ratio_threshold, size_diff_min_bytes):
                        print(f"↻ Size mismatch for {dest.name}: local={local_size:,}B vs remote={remote_size:,}B — redownloading")
                        try:
                            dest.unlink()
                        except FileNotFoundError:
                            pass
                        # fall through to download below
                    else:
                        stats.skipped_existing += 1
                        pbar.update(1)
                        pbar.set_postfix(checked=stats.files_checked, dl=stats.downloaded,
                                         re=stats.redownloaded, skip=stats.skipped_existing, err=stats.errors)
                        print(f"⏭️  Exists (size ok), skipping: {dest}")
                        continue

            # Download/export the file (fresh or after deletion)
            try:
                if export_mime:
                    download_file_by_id(fid, str(dest), mime_type_if_export=export_mime)
                    # treat re-export as a fresh download
                    stats.downloaded += 1
                elif mt.startswith("application/vnd.google-apps."):
                    # Other Google-native types (Docs/Slides/Drawings) – skip or add export logic
                    stats.skipped_native += 1
                    print(f"↪ Skipping unsupported native type ({mt}) for {name}")
                else:
                    download_file_by_id(fid, str(dest))
                    if dest.exists():  # we can infer 'redownload' vs 'download' from previous existence
                        # If we got here with an existing file, it was a redownload case
                        stats.redownloaded += 1
                    else:
                        stats.downloaded += 1
            except Exception as e:
                stats.errors += 1
                print(f"❌ Error on {name}: {e}")

            pbar.update(1)
            pbar.set_postfix(
                checked=stats.files_checked,
                dl=stats.downloaded,
                re=stats.redownloaded,
                skip=stats.skipped_existing,
                err=stats.errors,
            )

    if _root:
        print(
            "\nSummary:"
            f"\n  • Files checked:        {stats.files_checked}"
            f"\n  • Downloaded (fresh):   {stats.downloaded}"
            f"\n  • Redownloaded (fixed): {stats.redownloaded}"
            f"\n  • Skipped (existing):   {stats.skipped_existing}"
            f"\n  • Skipped (native):     {stats.skipped_native}"
            f"\n  • Errors:               {stats.errors}"
        )
    return stats


In [37]:
# --- FIX: rebuild get_authed_session without the unsupported `session=` kwarg,
#          still disabling proxies and adding a default timeout.
from google.auth.transport.requests import AuthorizedSession

def get_authed_session(creds):
    authed = AuthorizedSession(creds)
    # Ignore system/corporate proxies to avoid SSL handshake issues
    try:
        authed.trust_env = False
    except Exception:
        pass
    _orig_request = authed.request
    def _with_timeout(method, url, **kwargs):
        # Increase if you have very large files or slow links
        kwargs.setdefault("timeout", 300)
        return _orig_request(method, url, **kwargs)
    authed.request = _with_timeout
    return authed


# --- REPLACE sync_drive_folder with a version that classifies fresh vs re-downloads correctly.
#     (Assumes: _tqdm, list_children_all_drives, _decide_export_and_dest, GOOGLE_MIME_*,
#               drive_service, get_creds, download_file_by_id already exist.)
from dataclasses import dataclass
from pathlib import Path

try:
    _tqdm  # from tqdm import tqdm as _tqdm (text mode)
except NameError:
    from tqdm import tqdm as _tqdm

@dataclass
class SyncStats:
    files_checked: int = 0
    downloaded: int = 0
    redownloaded: int = 0
    skipped_existing: int = 0
    skipped_native: int = 0
    errors: int = 0

def _get_remote_info(svc, file_id: str):
    return svc.files().get(
        fileId=file_id,
        fields="size, md5Checksum, mimeType, modifiedTime, name",
        supportsAllDrives=True,
    ).execute()

def _significant_size_diff(local_size: int, remote_size: int | None,
                           ratio_threshold: float, min_bytes: int) -> bool:
    if remote_size is None or remote_size <= 0:
        return False
    diff = abs(local_size - remote_size)
    return diff >= min_bytes and (diff / max(remote_size, 1)) >= ratio_threshold

def sync_drive_folder(
    folder_id: str,
    local_dir: str,
    *,
    stats: SyncStats | None = None,
    svc=None,
    _root: bool = True,
    size_diff_ratio_threshold: float = 0.05,  # ≥5%
    size_diff_min_bytes: int = 1024,          # ≥1 KiB
    refresh_native_if_zero: bool = True       # re-export Sheets if local file is 0 B
):
    if stats is None:
        stats = SyncStats()
    if svc is None:
        svc = drive_service(get_creds())

    Path(local_dir).mkdir(parents=True, exist_ok=True)

    children = list_children_all_drives(svc, folder_id)
    file_items = [it for it in children if it["mimeType"] != GOOGLE_MIME_FOLDER]
    folder_name = Path(local_dir).name or str(local_dir)
    print(f"📂 {folder_name}: {len(children)} item(s) ({len(file_items)} files)")

    with _tqdm(total=len(file_items), desc=f"{folder_name}", unit="file", leave=True) as pbar:
        for it in children:
            name, fid, mt = it["name"], it["id"], it["mimeType"]

            if mt == GOOGLE_MIME_FOLDER:
                # Recurse into subfolder
                sync_drive_folder(
                    fid, str(Path(local_dir, name)),
                    stats=stats, svc=svc, _root=False,
                    size_diff_ratio_threshold=size_diff_ratio_threshold,
                    size_diff_min_bytes=size_diff_min_bytes,
                    refresh_native_if_zero=refresh_native_if_zero,
                )
                continue

            stats.files_checked += 1

            export_mime, dest = _decide_export_and_dest(name, mt, local_dir)
            was_existing = dest.exists()
            need_download = True  # assume we need to, then decide below

            if was_existing:
                if export_mime:
                    # Google Sheets export: no reliable remote size; refresh only if 0 B
                    if refresh_native_if_zero and dest.stat().st_size == 0:
                        print(f"↻ Local is 0 bytes, re-exporting (Google Sheet): {dest.name}")
                        try: dest.unlink()
                        except FileNotFoundError: pass
                        need_download = True
                    else:
                        need_download = False
                        stats.skipped_existing += 1
                        pbar.update(1)
                        pbar.set_postfix(checked=stats.files_checked, dl=stats.downloaded,
                                         re=stats.redownloaded, skip=stats.skipped_existing, err=stats.errors)
                        print(f"⏭️  Exists, skipping: {dest}")
                        continue
                else:
                    # Non-native file: compare sizes
                    local_size = dest.stat().st_size
                    remote_size = None
                    try:
                        meta = _get_remote_info(svc, fid)
                        if "size" in meta and meta["size"] is not None:
                            remote_size = int(meta["size"])
                    except Exception as e:
                        print(f"ℹ️  Could not fetch remote size for {name}: {e}")

                    if _significant_size_diff(local_size, remote_size,
                                              size_diff_ratio_threshold, size_diff_min_bytes):
                        print(f"↻ Size mismatch for {dest.name}: local={local_size:,}B vs remote={remote_size:,}B — redownloading")
                        try: dest.unlink()
                        except FileNotFoundError: pass
                        need_download = True
                    else:
                        need_download = False
                        stats.skipped_existing += 1
                        pbar.update(1)
                        pbar.set_postfix(checked=stats.files_checked, dl=stats.downloaded,
                                         re=stats.redownloaded, skip=stats.skipped_existing, err=stats.errors)
                        print(f"⏭️  Exists (size ok), skipping: {dest}")
                        continue

            # Perform download/export if needed
            did_download = False
            try:
                if need_download:
                    if export_mime:
                        download_file_by_id(fid, str(dest), mime_type_if_export=export_mime)
                        did_download = True
                    elif mt.startswith("application/vnd.google-apps."):
                        stats.skipped_native += 1
                        print(f"↪ Skipping unsupported native type ({mt}) for {name}")
                        did_download = False
                    else:
                        download_file_by_id(fid, str(dest))
                        did_download = True
                # update counters correctly
                if did_download:
                    if was_existing:
                        stats.redownloaded += 1
                    else:
                        stats.downloaded += 1
            except Exception as e:
                stats.errors += 1
                print(f"❌ Error on {name}: {e}")

            pbar.update(1)
            pbar.set_postfix(
                checked=stats.files_checked,
                dl=stats.downloaded,
                re=stats.redownloaded,
                skip=stats.skipped_existing,
                err=stats.errors,
            )

    if _root:
        print(
            "\nSummary:"
            f"\n  • Files checked:        {stats.files_checked}"
            f"\n  • Downloaded (fresh):   {stats.downloaded}"
            f"\n  • Redownloaded (fixed): {stats.redownloaded}"
            f"\n  • Skipped (existing):   {stats.skipped_existing}"
            f"\n  • Skipped (native):     {stats.skipped_native}"
            f"\n  • Errors:               {stats.errors}"
        )
    return stats


In [41]:
sync_drive_folder(
    '1bZYi9zgWendn5ZfqFCbigJ-fXuh7lb5_',
    r'D:\PhD Thesis Writing\Data\2025-11-26',
    size_diff_ratio_threshold=0.02,  # 2%
    size_diff_min_bytes=2048,        # 2 KiB
)

📂 2025-11-26: 6 item(s) (0 files)


2025-11-26: 0file [00:00, ?file/s]

📂 Day2LongNoFibre: 36 item(s) (36 files)



Day2LongNoFibre:   3%|▋                         | 1/36 [00:00<00:17,  2.03file/s, checked=1, dl=0, err=0, re=0, skip=1]

⏭️  Exists (size ok), skipping: D:\PhD Thesis Writing\Data\2025-11-26\Day2LongNoFibre\Day_2_1MhzGate_107uWperArm_Rmp1000Hz_voaramp_20251126T184351+0000.npz



Day2LongNoFibre:   6%|█▍                        | 2/36 [00:01<00:21,  1.58file/s, checked=2, dl=0, err=0, re=0, skip=2]

... (450 outputs left out)

✔ Saved: D:\PhD Thesis Writing\Data\2025-11-26\Night2LongNoFibre\Test_Night_2_1MhzGate_195uWperArm_Rmp1Hz_nomodulation_20251126T145518+0000.npz




Test_Night_2_1MhzGate_195uWperArm_Rmp1Hz_nomodulation_20251126T145518+0000_meta.txt: 100%|████| 473/473 [00:00<?, ?B/s]

Night2LongNoFibre: 100%|██████████████████| 42/42 [02:17<00:00,  3.28s/file, checked=228, dl=10, err=0, re=1, skip=217]
2025-11-26: 0file [05:36, ?file/s]

✔ Saved: D:\PhD Thesis Writing\Data\2025-11-26\Night2LongNoFibre\Test_Night_2_1MhzGate_195uWperArm_Rmp1Hz_nomodulation_20251126T145518+0000_meta.txt

Summary:
  • Files checked:        228
  • Downloaded (fresh):   10
  • Redownloaded (fixed): 1
  • Skipped (existing):   217
  • Skipped (native):     0
  • Errors:               0


SyncStats(files_checked=228, downloaded=10, redownloaded=1, skipped_existing=217, skipped_native=0, errors=0)

In [42]:
sync_drive_folder(
    '1AYVPHgaZNnKWR9arOYzfoWvtqdNvRwsm',
    r'D:\PhD Thesis Writing\Data\2025-11-28',
    size_diff_ratio_threshold=0.02,  # 2%
    size_diff_min_bytes=2048,        # 2 KiB
)

📂 2025-11-28: 3 item(s) (0 files)


2025-11-28: 0file [00:00, ?file/s]

📂 Night2LongVeryFast: 37 item(s) (37 files)



Night2LongVeryFast:   0%|                                                                     | 0/37 [00:00<?, ?file/s]

NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000.npz: 0.00B [00:00, ?B/s]

NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000.npz: 119kB [00:00, 1.04MB/s]

NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000.npz: 410kB [00:00, 1.73MB/s]

NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000.npz: 658kB [00:00, 1.90MB/s]

NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000.npz: 904kB [00:00, 1.90MB/s]

NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000.npz: 1.13MB [00:00, 2.01MB/s]

NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000.npz: 1.37MB [00:00, 2.15MB/s]

NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000.npz: 1.61MB [00:00, 2.14MB/s]

NightVeryFast_2_1MhzGate_147uWp

✔ Saved: D:\PhD Thesis Writing\Data\2025-11-28\Night2LongVeryFast\NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000.npz




NightVeryFast_2_1MhzGate_147uWperArm_Rmp1000Hz_voaramp_20251128T195221+0000_meta.txt: 100%|███| 481/481 [00:00<?, ?B/s]

Night2LongVeryFast:   5%|█▏                     | 2/37 [00:24<06:05, 10.45s/file, checked=2, dl=2, err=0, re=0, skip=0]

... (216 outputs left out)

✔ Saved: D:\PhD Thesis Writing\Data\2025-11-28\Night2Long\Night_2_1MhzGate_158uWperArm_Rmp1Hz_20251128T173005+0000_meta.txt




Night_2_1MhzGate_158uWperArm_Rmp1Hz_20251128T172516+0000.csv: 0.00B [00:00, ?B/s]

Night_2_1MhzGate_158uWperArm_Rmp1Hz_20251128T172516+0000.csv: 345kB [00:00, 3.52MB/s]

Night_2_1MhzGate_158uWperArm_Rmp1Hz_20251128T172516+0000.csv: 1.06MB [00:00, 5.82MB/s]

Night_2_1MhzGate_158uWperArm_Rmp1Hz_20251128T172516+0000.csv: 1.81MB [00:00, 6.73MB/s]

Night_2_1MhzGate_158uWperArm_Rmp1Hz_20251128T172516+0000.csv: 2.64MB [00:00, 6.08MB/s]

Night2Long: 100%|██████████████████████████| 37/37 [10:21<00:00, 16.80s/file, checked=111, dl=111, err=0, re=0, skip=0]
2025-11-28: 0file [31:43, ?file/s]

✔ Saved: D:\PhD Thesis Writing\Data\2025-11-28\Night2Long\Night_2_1MhzGate_158uWperArm_Rmp1Hz_20251128T172516+0000.csv

Summary:
  • Files checked:        111
  • Downloaded (fresh):   111
  • Redownloaded (fixed): 0
  • Skipped (existing):   0
  • Skipped (native):     0
  • Errors:               0


SyncStats(files_checked=111, downloaded=111, redownloaded=0, skipped_existing=0, skipped_native=0, errors=0)

In [ ]:
FOLDER_ID = '1bZYi9zgWendn5ZfqFCbigJ-fXuh7lb5_'
FILE_ID   = None
OUT_DIR   = r'D:\PhD Thesis Writing\Data\2025-11-26'
sync_drive_folder(FOLDER_ID, OUT_DIR)

In [ ]:
# 4) Configure & run: set FOLDER_ID or FILE_ID and output directory
FOLDER_ID = '18BMH-PhTZNiN8rbXyqZvz5J9qRGrRqbA'
FILE_ID   = None
OUT_DIR   = r'D:\PhD Thesis Writing\Data\2025-11-26'

from pathlib import Path
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# If you previously used custom CREDS_PATH/TOKEN_PATH with functools.partial, keep that as-is.

if FILE_ID:
    # Single file path example (adjust name/extension as needed)
    download_file_by_id(FILE_ID, str(Path(OUT_DIR, "downloaded_file")))
elif FOLDER_ID:
    sync_drive_folder(FOLDER_ID, OUT_DIR)
else:
    print("Set FOLDER_ID or FILE_ID first.")

📂 Data: 45 item(s) (0 files)


Data: 0file [00:00, ?file/s]

📂 24_11_2025: 3 item(s) (0 files)



24_11_2025: 0file [00:00, ?file/s]

📂 TestFolder: 3 item(s) (3 files)




TestFolder:   0%|                                                                              | 0/3 [00:00<?, ?file/s]


ni6366_binReq-10000Hz_binReal-10000Hz_AO-250000Hz_VOAreq-5.2Hz_VOAreal-5.19999Hz_mode-repeat_20251124T141106+0000.csv: 0.00B [00:00, ?B/s]


ni6366_binReq-10000Hz_binReal-10000Hz_AO-250000Hz_VOAreq-5.2Hz_VOAreal-5.19999Hz_mode-repeat_20251124T141106+0000.csv: 163kB [00:00, 1.32MB/s]


ni6366_binReq-10000Hz_binReal-10000Hz_AO-250000Hz_VOAreq-5.2Hz_VOAreal-5.19999Hz_mode-repeat_20251124T141106+0000.csv: 1.16MB [00:00, 5.81MB/s]


ni6366_binReq-10000Hz_binReal-10000Hz_AO-250000Hz_VOAreq-5.2Hz_VOAreal-5.19999Hz_mode-repeat_20251124T141106+0000.csv: 2.79MB [00:00, 9.93MB/s]


ni6366_binReq-10000Hz_binReal-10000Hz_AO-250000Hz_VOAreq-5.2Hz_VOAreal-5.19999Hz_mode-repeat_20251124T141106+0000.csv: 4.09MB [00:00, 9.52MB/s]


ni6366_binReq-10000Hz_binReal-10000Hz_AO-250000Hz_VOAreq-5.2Hz_VOAreal-5.19999Hz_mode-repeat_20251124T141106+0000.csv: 5.36MB [00:00, 7.71MB/s]


ni6366

... (514 outputs left out)

✔ Saved: C:\Users\Ry\Documents\PhD Thesis Writing\Data\24_07_2025\measurement_24_07_25_fan_lvts.npy




measurement_24_07_25_balanced2_lvns.npy: 0.00B [00:00, ?B/s]

measurement_24_07_25_balanced2_lvns.npy: 416kB [00:00, 2.20MB/s]

24_07_2025:  67%|█████████████████▌        | 29/43 [01:11<00:26,  1.89s/file, checked=256, dl=256, err=0, re=0, skip=0]

✔ Saved: C:\Users\Ry\Documents\PhD Thesis Writing\Data\24_07_2025\measurement_24_07_25_balanced2_lvns.npy




measurement_24_07_25_balanced2_lvts.npy: 0.00B [00:00, ?B/s]

measurement_24_07_25_balanced2_lvts.npy: 609kB [00:00, 5.81MB/s]

measurement_24_07_25_balanced2_lvts.npy: 1.86MB [00:00, 9.59MB/s]

measurement_24_07_25_balanced2_lvts.npy: 3.24MB [00:00, 11.7MB/s]

In [ ]:
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
FOLDER_ID = '18BMH-PhTZNiN8rbXyqZvz5J9qRGrRqbA'  # just the ID
DRIVE_ID = '0AHMeY0Fo1d79Uk9PVA'
CUSTOM_DOWNLOAD_PATH = r'C:\Users\Ry\Documents\PhD Thesis Writing\Data'

creds = authenticate()
service = build('drive', 'v3', credentials=creds)

files = collect_files(service, FOLDER_ID, CUSTOM_DOWNLOAD_PATH, DRIVE_ID)
download_all_files(service, files, max_workers=4)